In [1]:
# Clone your repository
!git clone https://github.com/ilatims-b/error_measurement.git

# Change working directory to the cloned repo
%cd error_measurement

# Install required packages
!pip install -q sec-edgar-downloader tiktoken openai transformers accelerate

Cloning into 'error_measurement'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 117 (delta 63), reused 75 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (117/117), 32.80 KiB | 5.47 MiB/s, done.
Resolving deltas: 100% (63/63), done.
/kaggle/working/error_measurement


In [2]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Retrieve the token from Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("hf")

# Login to Hugging Face
login(token=hf_token)

In [3]:
# !watch -n 3 nvidia-smi

In [4]:
# %%bash
# echo "=========================================="
# echo " Step 1: Downloading & Processing SEC Data"
# echo "=========================================="

# python src/dataset.py \
#     --email "ce24b119@smail.iitm.ac.in" \
#     --download-dir "./data/sec_filings" \
#     --output-file "./data/processed_dataset.json" \
#     --seed-tokens 1000

In [5]:
# %%bash
# echo "=========================================="
# echo " Step 2: Generating Text Continuations"
# echo "=========================================="

# python src/generate.py \
#     --input-file "./data/processed_dataset.json" \
#     --output-file "./data/generations.json" \
#     --model-name "meta-llama/Llama-3.2-3B-Instruct" \
#     --max-new-tokens 256 \
#     --num-continuations 1


## Step 2 (API): Generate Continuations via Grok

Uses the **Grok (xAI) API** instead of a local GPU model.  
Requires a `GROK_API_KEY` Kaggle secret (Add-ons → Secrets).  
Rate limiting is handled automatically via a sliding-window RPM + TPM tracker (same pattern as `fact_pipeline.py`).

**Defaults**: model=`grok-3-mini`, RPM=60, TPM=100 000.  
Override any variable in the cell below.

In [ ]:
# ── Grok API generation ─────────────────────────────────────────────────
# Pull the Grok API key from Kaggle Secrets (set this in Add-ons → Secrets)
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
grok_api_key = user_secrets.get_secret("GROK_API_KEY")

# ── Configurable parameters ──────────────────────────────────────────────
GROK_MODEL     = "grok-3-mini"          # or "grok-3", "grok-2", etc.
GROK_BASE_URL  = "https://api.x.ai/v1"  # xAI OpenAI-compatible endpoint
NUM_CONTS      = 3                       # continuations per document
MAX_NEW_TOKENS = 512
TEMPERATURE    = 0.7
TOP_P          = 0.9
RPM            = 60      # requests per minute — adjust to your Grok tier
TPM            = 100000  # tokens per minute   — adjust to your Grok tier

INPUT_FILE  = "./data/processed_dataset.json"
OUTPUT_FILE = "./data/generations.json"

# ── Run generation ───────────────────────────────────────────────────────
import subprocess, sys

cmd = [
    sys.executable, "src/generate.py",
    "--input-file",        INPUT_FILE,
    "--output-file",       OUTPUT_FILE,
    "--model-name",        GROK_MODEL,
    "--num-continuations", str(NUM_CONTS),
    "--max-new-tokens",    str(MAX_NEW_TOKENS),
    "--temperature",       str(TEMPERATURE),
    "--top-p",             str(TOP_P),
    "--api-key",           grok_api_key,
    "--base-url",          GROK_BASE_URL,
    "--rpm",               str(RPM),
    "--tpm",               str(TPM),
]

result = subprocess.run(cmd, check=True)
print(f"\nGeneration complete. Output saved to {OUTPUT_FILE}")

In [6]:
  # parser = argparse.ArgumentParser(description="Extract and verify factual claims using LLM APIs.")
  #   parser.add_argument("--gen-file", type=str, default="./data/generations.json", help="Input generations JSON")
  #   parser.add_argument("--source-file", type=str, default="./data/processed_dataset.json", help="Original source documents JSON")
  #   parser.add_argument("--output-file", type=str, default="./data/evaluated_generations.json", help="Output verification JSON")
  #   parser.add_argument("--api-key", type=str, required=True, help="API Key for OpenAI/Grok/Groq")
  #   parser.add_argument("--base-url", type=str, default="https://api.x.ai/v1", help="API base URL")
  #   parser.add_argument("--verification-model-name", type=str, default="qwen3-32b", help="Model name for extraction/verification")
  #   parser.add_argument("--extraction-model-name",type=str, default="llama-4-scout-17b", help="model for claim extraction")
  #   parser.add_argument("--chunk-size", type=int, default=128, help="Token size per evaluation chunk")
  #   parser.add_argument("--extraction-prompt", type=str, default=None, help="Custom prompt for claim extraction")
  #   parser.add_argument("--verification-prompt", type=str, default=None, help="Custom prompt for claim verification")
  #   parser.add_argument("--max-passage-tokens", type=int, default=5000, help="Max tokens per passage")
  #   args = parser.parse_args()

In [7]:
EXTRACTION_PROMPT = """You are a JSON-only extraction engine. You output nothing except a raw JSON array.

TASK: Extract verifiable factual claims from the financial text the user provides.

CLAIM TYPES:
- NUMERIC: Contains a specific number or statistic. The object field MUST be the actual number (e.g., "$4.53 billion", "62.4%", "$1.42B decrease").
- ENTITY: Named entity relationships (e.g., "Amgen acquired Teneobio for $2.85 billion").
- DIRECTIONAL: Relative change with magnitude. Object MUST include direction AND magnitude (e.g., "decreased 18% to $2.73B"). SKIP directional claims where no magnitude is given.

RULES:
- Omit FORWARD-looking claims (guidance, expectations, "we expect", "may", "could").
- Omit vague generic statements (e.g., "revenue may be affected by market conditions").
- Omit claims where the object is only a time period with no value.
- Output ONLY a raw JSON array. No prose, no markdown, no explanation, no ```json fences.
- If no valid claims exist, output: []

OUTPUT FORMAT (exactly):
[{"subject": "...", "relation": "...", "object": "...", "type": "NUMERIC|ENTITY|DIRECTIONAL"}]"""

In [8]:
VERIFICATION_PROMPT ="""You are a financial fact-checker. Your job is to verify a specific claim against a source document excerpt.

Source document excerpt:
{passage}

Claim to verify ({type}):
  Subject: {subject}
  Relation: {relation}
  Object: {object}

Instructions:
- Read the excerpt carefully. Look for any numbers, entities, or directional statements that relate to this claim.
- If the excerpt explicitly states or implies the claim is correct, answer A.
- If the excerpt explicitly states something that contradicts the claim, answer B.
- Only answer C if the excerpt contains NO relevant information whatsoever about this claim's subject.
- Do NOT answer C just because the exact phrasing differs — look for semantic equivalence.

Think step by step in one sentence, then give your final answer on a new line as exactly one letter: A, B, or C."""

In [ ]:
import subprocess, sys


result = subprocess.run([
    sys.executable, "src/fact_pipeline.py",
    "--gen-file", "/kaggle/input/datasets/smitali/error-measurement-data/generations_mid_company.jsonl",
    "--source-file", "/kaggle/input/datasets/smitali/error-measurement-data/processed_dataset_mid_companies.json",
    "--output-file", "./data/evaluated_generations.json",
    "--api-key", "",
    "--chunk-size", "128",
    "--base-url", "https://api.groq.com/openai/v1",
    "--extraction-model-name", "meta-llama/llama-4-scout-17b-16e-instruct",
    "--verification-model-name", "qwen/qwen3-32b",
    "--max-passage-tokens", " 5000",
    "--extraction-prompt", EXTRACTION_PROMPT,
    "--verification-prompt", VERIFICATION_PROMPT,
], capture_output=False)

2026-04-23 20:29:25,560 - __main__ - INFO - Starting fact extraction and verification pipeline...
2026-04-23 20:29:25,563 - __main__ - INFO - Processing document: data/sec_filings/sec-edgar-filings/CMCSA/10-Q/0001166691-23-000040/full-submission.txt
2026-04-23 20:29:27,073 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-23 20:29:27,095 - __main__ - INFO - Truncating source passage from 16707 to 5000 tokens to respect limits.
2026-04-23 20:29:27,782 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-23 20:29:27,793 - __main__ - INFO - Truncating source passage from 16707 to 5000 tokens to respect limits.
2026-04-23 20:29:27,796 - __main__ - INFO - Rate limit approaching for verification. Sleeping for 59.30 seconds...
2026-04-23 20:30:27,759 - httpx - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-23 20:30:27,769 - _

In [ ]:
# %%bash
# echo "=========================================="
# echo " Step 3: Fact Extraction and Verification"
# echo "=========================================="

# python src/fact_pipeline.py \
#     --gen-file "/kaggle/input/datasets/smitali/error-measurement-data/generations_top_companies.json" \
#     --source-file "/kaggle/input/datasets/smitali/error-measurement-data/processed_dataset_top_companies.json" \
#     --output-file "./data/evaluated_generations.json" \
#     --api-key "" \
#     --extraction-prompt  EXTRACTION_PROMPT\
#     --verification-prompt VERIFICATION_PROMPT \
#     --chunk-size 128\
#     --max-passage-tokens 5000\
#     --base-url "https://api.groq.com/openai/v1" \
#     --verification-model-name "qwen/qwen3-32b" \
#     --extraction-model-name "meta-llama/llama-4-scout-17b-16e-instruct" \